# Questão 5 — Análise de Clientes

## Objetivo
Identificar os clientes fiéis da LH Nautical, considerando não apenas alto gasto, mas também recorrência e diversidade de categorias consumidas.

## Regras de negócio aplicadas
- **Faturamento Total**: soma do valor vendido por cliente
- **Frequência**: contagem de transações distintas por cliente
- **Ticket Médio**: faturamento total dividido pela frequência
- **Diversidade de Categorias**: quantidade de categorias distintas consumidas por cliente
- **Filtro de Elite**: apenas clientes com compras em **3 ou mais categorias**
- **Desempate**: em caso de empate no ticket médio, ordenar por `customer_id` crescente

## Estratégia adotada
A análise foi construída sobre o modelo dimensional já tratado no dbt:
- `fct_vendas`: fato principal com as transações
- `dim_produto`: dimensão com categoria padronizada
- `mart_clientes_fieis`: ranking final dos 10 clientes selecionados
- `mart_categoria_top10_clientes`: categoria mais consumida por esse grupo

In [0]:
%sql
-- Conferência inicial da fato de vendas
SELECT *
FROM lh_nautical.04_gold.fct_vendas
LIMIT 10;

In [0]:
%sql
-- Conferência inicial da dimensão de produtos
-- coluna "category" representa a categoria já padronizada para análise.
-- coluna "actual_category" preserva a grafia original para auditoria.
SELECT
    product_id,
    product_name,
    category,
    actual_category
FROM lh_nautical.04_gold.dim_produto
LIMIT 20;

## Etapa 2 — Cálculo dos indicadores por cliente

A consulta abaixo demonstra a lógica de negócio aplicada para:
- calcular faturamento total
- calcular frequência
- calcular ticket médio
- medir diversidade de categorias
- aplicar o filtro de elite
- selecionar os Top 10 clientes fiéis

In [0]:
%sql
-- Questão 5.1
-- Calcula:
-- 1) Ticket médio e diversidade de categorias por cliente
-- 2) Top 10 clientes fiéis (maior ticket médio entre clientes com diversidade >= 3)
-- 3) Categoria mais vendida em quantidade total de itens para esse grupo

WITH base_vendas AS (

    SELECT
        f.sale_id,
        f.customer_id,
        f.product_id,
        f.quantity,
        f.receita_transacao_brl,
        p.category
    FROM lh_nautical.04_gold.fct_vendas f
    LEFT JOIN lh_nautical.04_gold.dim_produto p
        ON f.product_id = p.product_id

),

indicadores_cliente AS (

    SELECT
        customer_id,

        -- Faturamento total do cliente
        SUM(receita_transacao_brl) AS faturamento_total,

        -- Frequência de compras
        COUNT(DISTINCT sale_id) AS frequencia,

        -- Ticket médio
        ROUND(SUM(receita_transacao_brl) / COUNT(DISTINCT sale_id), 2) AS ticket_medio,

        -- Diversidade de categorias compradas
        COUNT(DISTINCT category) AS diversidade_categorias

    FROM base_vendas
    GROUP BY customer_id

),

top_10_clientes_fieis AS (

    SELECT
        customer_id,
        faturamento_total,
        frequencia,
        ticket_medio,
        diversidade_categorias
    FROM indicadores_cliente
    WHERE diversidade_categorias >= 3
    ORDER BY ticket_medio DESC, customer_id ASC
    LIMIT 10

),

categoria_top_10 AS (

    SELECT
        b.category,
        SUM(b.quantity) AS quantidade_total_itens
    FROM base_vendas b
    INNER JOIN top_10_clientes_fieis t
        ON b.customer_id = t.customer_id
    GROUP BY b.category
    ORDER BY quantidade_total_itens DESC, category ASC
    LIMIT 1

)

SELECT
    t.customer_id,
    t.faturamento_total,
    t.frequencia,
    t.ticket_medio,
    t.diversidade_categorias,
    c.category AS categoria_mais_vendida_top_10,
    c.quantidade_total_itens AS qtd_itens_categoria_top_10
FROM top_10_clientes_fieis t
CROSS JOIN categoria_top_10 c
ORDER BY t.ticket_medio DESC, t.customer_id ASC;

In [0]:
# Consulta os dados das marts já criadas
df_top10 = spark.sql("""
SELECT *
FROM lh_nautical.04_gold.mart_clientes_fieis
ORDER BY ticket_medio DESC, customer_id ASC
""")

df_categoria = spark.sql("""
SELECT *
FROM lh_nautical.04_gold.mart_categoria_top10_clientes
ORDER BY quantidade_total_itens DESC, category ASC
LIMIT 1
""")

# Pega os valores principais
top_1 = df_top10.collect()[0]
categoria_top = df_categoria.collect()[0]

# Prints formatados
print("RESULTADO DA QUESTÃO 5\n")

print(f"Quantidade de clientes fiéis selecionados: {df_top10.count()}")

print(f"\nCliente com maior Ticket Médio: {top_1['customer_id']}")
print(f"Ticket Médio: {round(top_1['ticket_medio'], 2)}")
print(f"Diversidade de categorias: {top_1['diversidade_categorias']}")

print(f"\nCategoria mais vendida para os Top 10 clientes: {categoria_top['category']}")
print(f"Quantidade total de itens: {categoria_top['quantidade_total_itens']}")

## Etapa 3 — Uso da mart final de clientes fiéis

Como parte da modelagem analítica, a regra acima foi materializada em uma mart final.
Isso facilita a rastreabilidade e reaproveitamento da análise.

In [0]:
%sql
-- Resultado da seleção dos Top 10 clientes fiéis
SELECT *
FROM lh_nautical.04_gold.mart_clientes_fieis
ORDER BY ticket_medio DESC, customer_id ASC;

## Etapa 4 — Categoria mais comprada pelos Top 10 clientes

Após identificar os clientes fiéis, o próximo passo é descobrir
qual categoria concentrou a maior quantidade total de itens comprados por esse grupo.

In [0]:
%sql
-- Questão 5.1 (parte final)
-- Categoria mais vendida, em quantidade de itens, considerando apenas os Top 10 clientes selecionados

WITH top_10 AS (

    SELECT
        customer_id
    FROM lh_nautical.04_gold.mart_clientes_fieis

),

base AS (

    SELECT
        f.customer_id,
        f.quantity,
        p.category
    FROM lh_nautical.04_gold.fct_vendas f
    LEFT JOIN lh_nautical.04_gold.dim_produto p
        ON f.product_id = p.product_id
    INNER JOIN top_10 t
        ON f.customer_id = t.customer_id

),

categoria_agregada AS (

    SELECT
        category,
        SUM(quantity) AS quantidade_total_itens
    FROM base
    GROUP BY category

)

SELECT *
FROM categoria_agregada
ORDER BY quantidade_total_itens DESC, category ASC;

## Questão 5.3 — Explicação da lógica aplicada

### 1. Como foi realizada a limpeza das categorias?
A limpeza das categorias foi realizada anteriormente na preparação da dimensão de produtos.  
Os nomes inconsistentes e com erro de grafia foram consolidados em uma categoria padronizada, preservando a coluna original para rastreabilidade.  
Na análise final, foi utilizada a coluna `category`, já padronizada, garantindo que variações como grafias incorretas não gerassem duplicidade analítica.

### 2. Qual lógica foi utilizada para filtrar os clientes com diversidade mínima?
Para cada cliente, foi calculada a quantidade de categorias distintas compradas por meio de `COUNT(DISTINCT category)`.  
Em seguida, foi aplicado o critério de elite definido pela regra de negócio: somente clientes com diversidade de categorias maior ou igual a 3 foram mantidos no ranking.

### 3. Como foi garantido que a contagem de itens refletisse apenas os Top 10?
Primeiro foi gerada a lista final dos 10 clientes fiéis com base no maior ticket médio, respeitando o filtro de diversidade mínima e o desempate por `customer_id`.  
Depois, ao calcular a categoria mais vendida, a análise foi restrita exclusivamente a esse conjunto por meio de um `INNER JOIN` entre a fato de vendas e a mart dos Top 10 clientes.  
Dessa forma, a soma de `quantity` considerou apenas compras realizadas pelos clientes efetivamente selecionados.